# 환경설정

In [ ]:
# 인증

from google.colab import auth
auth.authenticate_user()
print("인증 완료")

In [ ]:
# 매직 로드

%load_ext google.cloud.bigquery

# stg_event_properties — 세션·유저 매핑 스테이징

**목적**: `event_properties`(세션↔유저 매핑 원본)를 분석용 Stage로 정제
**산출물**: `events.stg_event_properties` (VIEW)
**Grain**: 1행 = 1개 세션
**퍼널 역할**: session ↔ user 매핑 (다른 events 테이블과 유저를 연결할 때 반드시 이걸 거쳐야 함)

**이미 검증 완료된 사실 (재검증 불필요, 결과만 반영)**
- `event_properties.user_id`는 STRING → `SAFE_CAST(... AS INT64)` 필요
- `event_log` ↔ `event_properties` 세션 조인 성공률: **100%**
- 캐스팅된 user_id ↔ `raw_user.id` 매칭률: 로그인 유저 기준 **98.1%**
- 비로그인/게스트 세션(캐스팅 실패)은 전체의 약 29.5% — 여기서 제외하지 않고 플래그만 남김

**핵심 결정**
- `user_id`: STRING → INT64 캐스팅 (`SAFE_CAST`, 실패시 NULL)
- `is_logged_in`: user_id 캐스팅 성공 여부
- `is_user_matched`: 캐스팅된 user_id가 `stg_votes__user`(정제된 유저 테이블)에 실제로 존재하는지


In [ ]:
%%bigquery df_combo_check --project your-gcp-project

SELECT
  COUNT(*) AS row_count,
  COUNT(DISTINCT CONCAT(session_id, '|', COALESCE(user_id, ''))) AS distinct_combo_count
FROM `your-gcp-project.events.event_properties`;

In [ ]:
df_combo_check

In [ ]:
%%bigquery df_combo_dup_sample --project your-gcp-project

SELECT session_id, user_id, COUNT(*) AS cnt
FROM `your-gcp-project.events.event_properties`
GROUP BY session_id, user_id
HAVING COUNT(*) > 1
ORDER BY cnt DESC
LIMIT 5

In [ ]:
df_combo_dup_sample

In [ ]:
%%bigquery df_combo_dup_detail --project your-gcp-project

SELECT *
FROM `your-gcp-project.events.event_properties`
WHERE session_id = (
  SELECT session_id
  FROM `your-gcp-project.events.event_properties`
  GROUP BY session_id, user_id
  HAVING COUNT(*) > 1
  ORDER BY COUNT(*) DESC
  LIMIT 1
)

In [ ]:
df_combo_dup_detail

In [ ]:
%%bigquery df_id_order_check --project your-gcp-project

SELECT id, session_id, user_id, osversion, device_id
FROM `your-gcp-project.events.event_properties`
WHERE session_id = 'f03TUjX4YOO9Zp7mNluK6LXZEG52'
ORDER BY id

In [ ]:
df_id_order_check

## Raw 품질 확인

### 1. 컬럼 타입 확인

In [ ]:
%%bigquery df_types --project your-gcp-project

SELECT column_name, data_type
FROM `your-gcp-project.events.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'event_properties'
ORDER BY ordinal_position

In [ ]:
df_types

### 2. PK(session_id) 유니크 · 결측 확인

In [ ]:
%%bigquery df_props_pk_check --project your-gcp-project

SELECT
  COUNT(*) AS row_count,
  COUNT(DISTINCT session_id) AS distinct_session_id_count,
  COUNT(*) - COUNT(DISTINCT session_id) AS duplicate_session_id_count,
  COUNTIF(session_id IS NULL) AS null_session_id_count
FROM `your-gcp-project.events.event_properties`;

In [ ]:
df_props_pk_check

### 3. 전체 컬럼 결측 확인

In [ ]:
%%bigquery df_props_null_check --project your-gcp-project

SELECT
  COUNT(*) AS row_count,
  COUNTIF(user_id IS NULL) AS null_user_id,
  COUNTIF(language IS NULL) AS null_language,
  COUNTIF(osname IS NULL) AS null_osname,
  COUNTIF(osversion IS NULL) AS null_osversion,
  COUNTIF(versionname IS NULL) AS null_versionname,
  COUNTIF(device_id IS NULL) AS null_device_id
FROM `your-gcp-project.events.event_properties`;

In [ ]:
df_props_null_check.T.rename(columns={0: 'null_count'})

### 4. user_id 캐스팅 성공/실패 비율 (참고용, 이미 검증된 수치와 비교)

In [ ]:
%%bigquery df_cast_check --project your-gcp-project

SELECT
  COUNTIF(SAFE_CAST(user_id AS INT64) IS NULL) AS cast_fail_count,
  COUNTIF(SAFE_CAST(user_id AS INT64) IS NOT NULL) AS cast_success_count
FROM `your-gcp-project.events.event_properties`;

In [ ]:
df_cast_check

In [ ]:
%%bigquery df_id_user_check --project your-gcp-project

SELECT
  id,
  user_id,
  session_id,
  SAFE_CAST(user_id AS INT64) AS user_id_casted
FROM `your-gcp-project.events.event_properties`
WHERE SAFE_CAST(user_id AS INT64) IS NULL
ORDER BY id DESC
LIMIT 20

In [ ]:
df_id_user_check

In [ ]:
%%bigquery df_blank_check --project your-gcp-project

SELECT
  id,
  user_id,
  LENGTH(user_id) AS user_id_length,
  CASE
    WHEN user_id IS NULL THEN 'NULL'
    WHEN user_id = '' THEN '빈 문자열'
    WHEN TRIM(user_id) = '' THEN '공백만 있음'
    ELSE '값 있음'
  END AS blank_type
FROM `your-gcp-project.events.event_properties`
WHERE SAFE_CAST(user_id AS INT64) IS NULL
ORDER BY id DESC
LIMIT 50

In [ ]:
%%bigquery df_blank_count --project your-gcp-project

SELECT
  CASE
    WHEN user_id IS NULL THEN 'NULL'
    WHEN user_id = '' THEN '빈 문자열'
    WHEN TRIM(user_id) = '' THEN '공백만 있음'
    WHEN SAFE_CAST(user_id AS INT64) IS NULL THEN '기타 변환실패 (문자 포함 등)'
    ELSE '정상'
  END AS user_id_status,
  COUNT(*) AS cnt
FROM `your-gcp-project.events.event_properties`
GROUP BY user_id_status
ORDER BY cnt DESC

In [ ]:
df_blank_count

### 5. osname / language 분포 (참고용, 이상치 확인)

In [ ]:
%%bigquery df_os_dist --project your-gcp-project

SELECT osname, COUNT(*) AS cnt
FROM `your-gcp-project.events.event_properties`
GROUP BY osname
ORDER BY cnt DESC

In [ ]:
df_os_dist

## 최종 정제 미리보기

In [ ]:
%%bigquery df_props_preview --project your-gcp-project

WITH casted AS (
  SELECT
    session_id,
    SAFE_CAST(user_id AS INT64) AS user_id,
    language, osname, osversion, versionname, device_id
  FROM `your-gcp-project.events.event_properties`
)
SELECT
  c.session_id,
  c.user_id,
  c.user_id IS NOT NULL AS is_logged_in,
  u.user_id IS NOT NULL AS is_user_matched,   -- stg_votes__user에 실제로 존재하는지
  c.language,
  c.osname,
  c.osversion,
  c.versionname,
  c.device_id
FROM casted c
LEFT JOIN `your-gcp-project.stage.stg_votes__user` u
  ON c.user_id = u.user_id
LIMIT 30;

In [ ]:
df_props_preview.head(5)

## 최종 Stage

### view (관례)

In [ ]:
%%bigquery --project your-gcp-project
CREATE OR REPLACE VIEW `your-gcp-project.stage.stg_event_properties` AS
WITH casted AS (
  SELECT
    id,
    session_id,
    user_id AS raw_user_id,                      -- 원본 문자열 그대로 보존
    SAFE_CAST(user_id AS INT64) AS user_id,       -- 조인용 캐스팅 결과
    language, osname, osversion, versionname, device_id
  FROM `your-gcp-project.events.event_properties`
)
SELECT
  c.id,
  c.session_id,
  c.raw_user_id,
  c.user_id,
  c.user_id IS NOT NULL AS is_logged_in,
  u.user_id IS NOT NULL AS is_user_matched,
  COUNT(DISTINCT c.user_id) OVER (PARTITION BY c.session_id) > 1 AS is_session_ambiguous,
  c.language,
  c.osname,
  c.osversion,
  c.versionname,
  c.device_id
FROM casted c
LEFT JOIN `your-gcp-project.stage.stg_users` u
  ON c.user_id = u.user_id;

### 생성 후 검증

In [ ]:
%%bigquery df_verify --project your-gcp-project

SELECT
  -- 1. 기본 행 수 검증 (원본 그대로 다 살아있는지)
  COUNT(*) AS total_rows,
  COUNT(DISTINCT id) AS distinct_id,

  -- 2. 로그인 여부 분포
  COUNTIF(is_logged_in) AS logged_in_rows,
  COUNTIF(NOT is_logged_in) AS not_logged_in_rows,

  -- 3. 유저 매칭 검증 (로그인된 것 중 실제 유저인 비율)
  COUNTIF(is_logged_in AND is_user_matched) AS matched_rows,
  ROUND(
    COUNTIF(is_logged_in AND is_user_matched) / NULLIF(COUNTIF(is_logged_in), 0) * 100, 1
  ) AS match_rate_pct,                                  -- 이전 검증한 98.1%와 비슷해야 정상

  -- 4. 세션 신뢰도 검증
  COUNTIF(is_session_ambiguous) AS ambiguous_rows,
  ROUND(COUNTIF(is_session_ambiguous) / COUNT(*) * 100, 1) AS ambiguous_pct  -- 60%대로 나와야 정상

FROM `your-gcp-project.events.stg_event_properties`;